In [1]:
import kaggle

!kaggle datasets download ankitbansal06/retail-orders -f orders.csv

Dataset URL: https://www.kaggle.com/datasets/ankitbansal06/retail-orders
License(s): CC0-1.0




  0%|          | 0.00/200k [00:00<?, ?B/s]
100%|##########| 200k/200k [00:00<00:00, 273kB/s]
100%|##########| 200k/200k [00:00<00:00, 273kB/s]


In [2]:
#extract file from zip file
import zipfile
zip_ref = zipfile.ZipFile('orders.csv.zip') 
zip_ref.extractall() # extract file to dir
zip_ref.close() # close file

In [3]:
#read data from the file and handle null values
import pandas as pd
df = pd.read_csv('orders.csv',na_values=['Not Available','unknown'])
df['Ship Mode'].unique()

array(['Second Class', 'Standard Class', nan, 'First Class', 'Same Day'],
      dtype=object)

In [4]:
#rename columns names ..make them lower case and replace space with underscore
df.rename(columns={'Order Id':'order_id', 'City':'city'})
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(' ','_')
df.head(5)

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,cost_price,list_price,quantity,discount_percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5


In [5]:
#derive new columns discount , sale price and profit
df['discount']=df['list_price']*df['discount_percent']*.01
df['sale_price']= df['list_price']-df['discount']
df['profit']=df['sale_price']-df['cost_price']
df

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,cost_price,list_price,quantity,discount_percent,discount,sale_price,profit
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2,5.2,254.8,14.8
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3,21.9,708.1,108.1
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5,0.5,9.5,-0.5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2,19.2,940.8,160.8
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5,1.0,19.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,2023-02-18,Second Class,Consumer,United States,Miami,Florida,33180,South,Furniture,Furnishings,FUR-FU-10001889,30,30,3,4,1.2,28.8,-1.2
9990,9991,2023-03-17,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Furniture,Furnishings,FUR-FU-10000747,70,90,2,4,3.6,86.4,16.4
9991,9992,2022-08-07,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Technology,Phones,TEC-PH-10003645,220,260,2,2,5.2,254.8,34.8
9992,9993,2022-11-19,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Office Supplies,Paper,OFF-PA-10004041,30,30,4,3,0.9,29.1,-0.9


In [6]:
#convert order date from object data type to datetime
df['order_date']=pd.to_datetime(df['order_date'],format="%Y-%m-%d")

In [7]:
#drop cost price list price and discount percent columns
df.drop(columns=['list_price','cost_price','discount_percent'],inplace=True)

In [8]:
df.head()

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,quantity,discount,sale_price,profit
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,2,5.2,254.8,14.8
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,3,21.9,708.1,108.1
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,2,0.5,9.5,-0.5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,5,19.2,940.8,160.8
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,2,1.0,19.0,-1.0


In [9]:
#load the data into sql server using replace option
import sqlalchemy as sal
engine = sal.create_engine('mssql://DESKTOP-M3R93F9/MISC?driver=ODBC+DRIVER+17+FOR+SQL+SERVER')
conn=engine.connect()

In [10]:
#load the data into sql server using append option
df.to_sql('df_orders', con=conn , index=False, if_exists = 'append')

38

## Data Analysis

In [11]:
# find top 10 highest reveue generating products
top_10_prod = df.groupby('product_id')['sale_price'].sum().reset_index()
top_10_prod = top_10_prod.sort_values(by='sale_price', ascending=False)
top_10_prod.head(10)

,product_id,sale_price
1614,TEC-CO-10004722,59514.0
776,OFF-BI-10003527,26525.3
1642,TEC-MA-10002412,21734.4
80,FUR-CH-10002024,21096.2
691,OFF-BI-10001359,19090.2
657,OFF-BI-10000545,18249.0
1604,TEC-CO-10001449,18151.2
1631,TEC-MA-10001127,17906.4
845,OFF-BI-10004995,17354.8
1420,OFF-SU-10000151,16325.8


In [12]:
# find top 5 highest selling products in each region
top_5_prod = df.groupby(['region', 'product_id'])['sale_price'].sum().reset_index()

top_5_prod['rn'] = top_5_prod.groupby('region')['sale_price'].rank(method='first', ascending=False)
top_5_products_region = top_5_prod[top_5_prod['rn'] <= 5]
top_5_products_region

,region,product_id,sale_price,rn
469,Central,OFF-BI-10000545,10132.7,4.0
488,Central,OFF-BI-10001120,11056.5,3.0
617,Central,OFF-BI-10004995,8416.1,5.0
1166,Central,TEC-CO-10004722,16975.0,1.0
1168,Central,TEC-MA-10000822,13770.0,2.0
1342,East,FUR-BO-10004834,11274.1,3.0
1834,East,OFF-BI-10001359,8463.6,4.0
2548,East,TEC-CO-10001449,8316.0,5.0
2556,East,TEC-CO-10004722,29099.0,1.0
2565,East,TEC-MA-10001047,13767.0,2.0


In [13]:
# find month over month growth comparison for 2022 and 2023 sales eg : jan 2022 vs jan 2023
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month

monthly_sales = df.groupby(['order_year', 'order_month'])['sale_price'].sum().reset_index()

monthly_comparison = monthly_sales.pivot(index='order_month', columns='order_year', values='sale_price').reset_index()
monthly_comparison.columns = ['order_month', 'sales_2022', 'sales_2023']
monthly_comparison = monthly_comparison.sort_values(by='order_month')
monthly_comparison

,order_month,sales_2022,sales_2023
0,1,94712.5,88632.6
1,2,90091.0,128124.2
2,3,80106.0,82512.3
3,4,95451.6,111568.6
4,5,79448.3,86447.9
5,6,94170.5,68976.5
6,7,78652.2,90563.8
7,8,104808.0,87733.6
8,9,79142.2,76658.6
9,10,118912.7,121061.5


In [14]:
# for each category which month had highest sales
df['order_year_month'] = df['order_date'].dt.strftime('%Y%m')

category_sales = df.groupby(['category', 'order_year_month'])['sale_price'].sum().reset_index()

category_sales['rn'] = category_sales.groupby('category')['sale_price'].rank(method='first', ascending=False)
highest_sales_month = category_sales[category_sales['rn'] == 1]
highest_sales_month

,category,order_year_month,sale_price,rn
9,Furniture,202210,42888.9,1.0
37,Office Supplies,202302,44118.5,1.0
69,Technology,202310,53000.1,1.0


In [15]:
# which sub category had highest growth by profit in 2023 compare to 2022
df['order_year'] = df['order_date'].dt.year

sub_category_sales = df.groupby(['sub_category', 'order_year'])['sale_price'].sum().reset_index()

sales_comparison = sub_category_sales.pivot(index='sub_category', columns='order_year', values='sale_price').reset_index()
sales_comparison.columns = ['sub_category', 'sales_2022', 'sales_2023']
sales_comparison['growth'] = sales_comparison['sales_2023'] - sales_comparison['sales_2022']

highest_growth_subcategory = sales_comparison.sort_values(by='growth', ascending=False).head(1)
highest_growth_subcategory

,sub_category,sales_2022,sales_2023,growth
11,Machines,73723.2,109178.5,35455.3
